# IFEval smoke run: judge-free grading through the ScreamingFace SDK

IFEval (arXiv:2311.07911) carries 541 prompts with machine-checkable constraints — word
counts, forbidden punctuation, required sections. The Engine grades every response with a
deterministic verifier: **no judge model, zero grading cost**.

This notebook runs the **corrective** method (the default): a bounded retry loop where
the checker's violations feed each retry — the protocol of
[*Beyond Leaderboards: Tokenomics of Agentic Small Language Model Ensembles*](https://openreview.net/forum?id=XSIYfTm2h7)
(Skurikhin et al., Los Alamos National Laboratory). Model calls are the only spend;
discovery and grading are free.

## Before running

The local AI Gateway must be running on `127.0.0.1:9105`, and the isolated Engine demo must be
running on `127.0.0.1:9108`. The connection panel sends the OpenRouter key through the Engine to
AI Gateway; the Client never calls AI Gateway directly.

For a host-local Engine, prepare IFEval's pinned cases (this also downloads the offline
NLTK tokenizer corpus the verifier reads) and pass the assets root explicitly:

```bash
uv run --with datasets python -m url4_cloud.benchmarks.ifeval.prepare \
  --out /tmp/screamingface-benchmark-assets/ifeval
URL4_BENCHMARK_ASSETS=/tmp/screamingface-benchmark-assets \
  uv run url4-cloud serve --local
```

`/opt/benchmarks` is the container image default and normally does not exist on the host.

In [ ]:
import screamingface as sf

## Connect OpenRouter

In [ ]:
sf.connect()

## Meet the benchmark

Before spending anything, read what the exam actually is. Discovery is free — plain
Engine REST, no model calls.

In [ ]:
sf.benchmarks.list()

In [ ]:
ifeval = sf.benchmarks.get("ifeval")
ifeval

### Read real prompts

Each prompt carries its constraints **in its own text** — "no commas", "at least 300
words", "highlight 3 sections". That is what makes IFEval machine-checkable: the Engine's
deterministic verifier re-reads the response against exactly those constraints, so
grading needs no judge model. Page further with `ifeval.cases(limit=3, offset=100)`.

In [ ]:
ifeval.cases(limit=3)

## Define a Candidate

In [ ]:
haiku = sf.Model("openrouter/anthropic/claude-haiku-4.5")

## Evaluate — the corrective chain (the default method)

IFEval here has two **methods** (the catalog entry above lists them):

- **`corrective`** (default) — the retry loop from
  [Skurikhin et al. (Los Alamos National Laboratory)](https://openreview.net/forum?id=XSIYfTm2h7),
  unrolled: the candidate answers, the deterministic checker grades it, the checker's
  *violations* are fed back, and the candidate retries — up to 3 attempts per prompt.
- **`single_pass`** — the paper's protocol: one answer, one check. This is the only
  score comparable to published IFEval numbers.

> **Cost note:** the chain is unrolled, so `limit=3` spends **9 candidate calls**
> (3 prompts × 3 attempts — every attempt runs even when the first one passed).
> Grading is still free.

Data flow (the exam owns the loop; haiku is just the answerer):

```text
SDK ── GET /v1/benchmarks/ifeval ──▶ Engine returns the CORRECTIVE url4
SDK links haiku into the one /candidate slot, submits ONE url4

per case (×3):
  haiku answers ─────────────────────▶ /check ▶ record 1
  haiku + answer 1 + verdict 1 ──────▶ /check ▶ record 2   (retry sees violations)
  haiku + answer 2 + verdict 2 ──────▶ /check ▶ record 3   (always runs — unrolled)

/aggregate: earliest strict-passing attempt is the case's answer
  ▶ score (retry protocol) + pass_at_1/2/3 + corrected_cases
```

In [ ]:
report = sf.evaluate(
    haiku,
    benchmark="ifeval",
    limit=3,
)
report

### Observe 1 — pass@attempt: two experiments hiding in one run

- `pass_at_1` is the **single-pass baseline**: the fraction of prompts the model
  nailed on its first try. This is the number you would compare (informally) to
  published IFEval scores.
- `pass_at_3` is what the corrective loop achieves — the headline `score`.
- `corrected_cases` counts prompts the checker's feedback actually **saved**: failed
  at attempt 1, passed later. If `pass_at_1 == pass_at_3`, the model needed no help
  on this slice — try a bigger `limit` or a smaller model to see the loop earn its
  keep.

In [ ]:
report.candidates[0].metrics

### Observe 2 — the experiment protocol is readable

The Engine sent back the *entire experiment* as one url4 expression before anything
ran — `report.candidates[0].url4` is the exact plan that executed. You can audit the
loop right in the string: three candidate slots, three checker calls, and the retry
prompt threading the previous verdict forward.

In [ ]:
plan = report.candidates[0].url4
print("attempts per case       :", plan.count("/candidate("))
print("checker calls per case  :", plan.count("/check("))
print("retry sees the verdict  :", "Checker verdict (JSON)" in plan)
print("exam revision in routes :", report.benchmark.revision in plan)

### Observe 3 — what the loop costs

Run the same slice with `method="single_pass"` and compare: the corrective run burns
roughly 3× the tokens for whatever accuracy it buys. (This is why corrective scores
must never sit next to published single-pass numbers — different protocol, different
budget.)

In [ ]:
baseline = sf.evaluate(haiku, benchmark="ifeval", limit=3, method="single_pass")
{
    "corrective": {
        "score": report.candidates[0].score,
        "output_tokens": report.usage.output_tokens,
    },
    "single_pass": {
        "score": baseline.candidates[0].score,
        "output_tokens": baseline.usage.output_tokens,
    },
}

### Observe 4 — peek at the raw execution stream (optional)

Every run streams events while it executes. Today they describe raw url4 node
lifecycle (semantic events — "case 3, attempt 2" — are in flight engine-side), but
even the raw counts show the machine at work: one evaluation fans out into dozens of
nodes, and only the model calls cost anything.

In [ ]:
from collections import Counter

events = []
sf.evaluate(haiku, benchmark="ifeval", limit=1, on_event=events.append, progress=False)
Counter(getattr(event, "name", None) or event.kind for event in events)

## The verifying ensemble — `sf.CorrectiveEnsemble`

The corrective method above put the retry loop in the BENCHMARK. The philosophically
clean home for it is the CANDIDATE — and that is exactly the system of
[Skurikhin et al. (Los Alamos National Laboratory)](https://openreview.net/forum?id=XSIYfTm2h7):
small-model members answer in parallel, the benchmark's own checker grades every
draft mid-flight, violations feed each member's retry (3 bounded attempts), and a
judge model tie-breaks among passers — with deterministic engine actions returning
the winner verbatim, so the judge cannot mutate it.

Because the loop lives inside the candidate, it runs against the frozen
**`single_pass`** exam — the ensemble and a solo model land in the SAME comparable
column.

> **Cost note:** per case the ensemble spends up to 3 members × 3 attempts + 3 judge
> calls = 12 model calls (checking is free). `limit=2` below ≈ 24 small-model calls
> plus 2 solo-model calls.

Data flow (the exam is identical for both rows; only the candidate differs):

```text
exam, per case:  /candidate(input, case) ──▶ ONE final answer ──▶ /check ▶ record
/aggregate: plain single-pass score — paper-comparable

row 1  haiku:     prompt ──▶ haiku ──▶ answer                     (1 call/case)

row 2  ensemble — everything below happens INSIDE the /candidate slot:
  attempt (×3):   kimi ─┐
                  deepseek ─┼─▶ each draft ──▶ /check ▶ feedback (violations text)
                  qwen ─┘
                  judge reads drafts + verdicts ──▶ letter ──▶ /select (verbatim)
  /finalize: earliest PASSED selection ──▶ the ONE answer the exam sees
                                                                 (12 calls/case)
```

In [ ]:
kimi = sf.Model("openrouter/moonshotai/kimi-k2.6")
deepseek = sf.Model("openrouter/deepseek/deepseek-v4-pro")
qwen = sf.Model("openrouter/qwen/qwen3.6-plus")
flash = sf.Model("openrouter/google/gemini-3-flash-preview")

ensemble = sf.CorrectiveEnsemble([kimi, deepseek, qwen], judge=flash)
ensemble

In [ ]:
duel = sf.evaluate(
    [haiku, ensemble],
    benchmark="ifeval",
    limit=2,
    method="single_pass",
)
duel

**What to observe:** both rows are the same exam and the same protocol — a fair
fight. The ensemble's `score` is what verification-and-retry buys; its
`output_tokens` next to the solo model's is the token overhead it costs. That
accuracy-vs-tokenomics tradeoff is the paper's whole argument, reproduced in one
dict:

In [ ]:
{
    candidate.name: {
        "score": candidate.score,
        "output_tokens": candidate.usage.output_tokens,
    }
    for candidate in duel.candidates
}

## Inspect the full Report

In [ ]:
report.candidates

In [ ]:
report.usage

In [ ]:
report.to_json()